In [144]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [145]:
df = pd.read_csv("Bengaluru_House_Data.xls")
df.head()
df.shape[0]
df.shape[1]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 936.7 KB


In [146]:
df.isnull().sum()
df['total_sqft']   =    df['total_sqft'].fillna("unknown")
df['bath']         =    df['bath'].fillna(df['bath'].mean())
df['balcony']      =    df['balcony'].fillna(df['balcony'].mean()) 
df['society']      =    df['society'].fillna("Unknown")
df['location']     =    df['location'].fillna("unknown")
df['size']         =    df['size'].fillna("unknown")

In [147]:
q1 = df['price'].quantile(0.25)
q2 = df['price'].quantile(0.75)
iqr = q2-q1
lower_bound = q1-iqr*1.5
upper_bound = q2+1.5*iqr
df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]
df.isnull().sum()


area_type       0
availability    0
location        0
size            0
society         0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

In [148]:
df['bhk'] = df['size'].str.extract('(\\d+)').astype(float)
df['availability'] = np.where(df['availability'] == 'Ready To Move', 1, 0)
df = df.drop(['size','society'],axis=1)
df = pd.get_dummies(df, columns=['area_type', 'location'])
df

,availability,total_sqft,bath,balcony,price,bhk,area_type_Built-up Area,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area,...,"location_ravindra nagar, T.dasarahalli peenya",location_rr nagar,location_sankeswari,location_sapthagiri Layout,location_singapura paradise,location_t.c palya,location_tc.palya,location_unknown,location_vinayakanagar,location_whitefiled
0,0,1056,2.0,1.0,39.07,2.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,1,2600,5.0,3.0,120.00,4.0,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
2,1,1440,2.0,3.0,62.00,3.0,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1,1521,3.0,1.0,95.00,3.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,1,1200,2.0,1.0,51.00,2.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13312,1,1262,2.0,2.0,47.00,2.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
13313,1,1345,2.0,1.0,57.00,3.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
13314,1,1715,3.0,3.0,112.00,3.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
13317,1,1141,2.0,1.0,60.00,2.0,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [149]:
X = df.drop('price', axis=1)  # All columns except price
y = df['price']               # Only price column
# Step 2: Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 4: Predict
y_pred = model.predict(X_test)

# Step 5: Evaluate
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("="*50)
print("LINEAR REGRESSION - BANGALORE HOUSE PRICE")
print("="*50)
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.2f} Lakhs")
print()

# Which features matter most?
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.coef_
}).sort_values('Importance', key=abs, ascending=False)

print("Most important features:")
print(importance.head(5))


ValueError: could not convert string to float: '300Sq. Yards'